# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, inspect, and analyze the FAIR^2 dataset using the `mlcroissant` library. We will step through data introspection, extraction, and perform basic exploratory data analysis (EDA).

### Dataset Source
The dataset is described using the Croissant schema, accessible at the following URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

print(f"Dataset name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Let's enumerate available record sets and their fields. We will use their `@id` identifiers in later steps.

In [ ]:
# List all record sets and their fields by @id
print("Available record sets (by @id):")
for recset in dataset.metadata.record_sets:
    print(f"- Record set @id: {recset['@id']} (name: {recset.get('name', 'N/A')})")
    if 'fields' in recset:
        for field in recset['fields']:
            print(f"    - Field @id: {field['@id']} (name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')})")
    elif 'columns' in recset:
        for col in recset['columns']:
            print(f"    - Column @id: {col['@id']} (name: {col.get('name', 'N/A')}, dataType: {col.get('dataType', 'N/A')})")

## 3. Data Extraction
Load data from one or more record sets using their `@id` as collected above. We'll place the data for each record set into a pandas DataFrame for analysis.

In [ ]:
# Extract data from all record sets
# Collect all record set @ids
record_sets = [recset['@id'] for recset in dataset.metadata.record_sets]
dataframes = {}
for record_set in record_sets:
    print(f"Loading records for record set: {record_set}")
    try:
        records = list(dataset.records(record_set=record_set))
        if records:
            df = pd.DataFrame(records)
            print(f"  Loaded with shape: {df.shape}")
            print(f"  Columns: {df.columns.tolist()}")
            dataframes[record_set] = df
        else:
            print(f"  No records found for @id {record_set}")
    except Exception as e:
        print(f"  Failed to load {record_set}: {e}")
# If at least one table loaded, preview it
if dataframes:
    main_recset = next(iter(dataframes.keys()))
    print(f"\nPreviewing first few records from record set: {main_recset}")
    display(dataframes[main_recset].head())

## 4. Exploratory Data Analysis (EDA)
Let's choose a numeric field from one record set for further analysis. We'll filter, normalize, and group as a demonstration.

In [ ]:
# For this demo, we'll find the first numeric column in the main record set and use it
def find_numeric_field(df):
    for col in df.columns:
        try:
            df[col].astype(float)
            return col
        except Exception:
            continue
    return None

if dataframes:
    import numpy as np
    main_recset = next(iter(dataframes.keys()))
    df = dataframes[main_recset]
    numeric_field = find_numeric_field(df)

    if numeric_field is not None:
        # Filtering: only keep values > median
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].median()
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > median ({threshold}): {len(filtered_df)} records")
        
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (mean=0, std=1):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Group by first non-numeric field
        non_numeric_cols = [c for c in df.columns if c != numeric_field and not np.issubdtype(df[c].dtype, np.number)]
        if non_numeric_cols:
            group_field = non_numeric_cols[0]
            print(f"\nGrouping by field: {group_field}")
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            display(grouped.head())
        else:
            print("No non-numeric fields for grouping found.")
    else:
        print("No numeric fields available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and a grouped aggregate if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    # If grouped data exists from the previous step
    if 'grouped' in locals():
        plt.figure(figsize=(10,4))
        sns.barplot(x=grouped[group_field], y=grouped[numeric_field])
        plt.title(f"Average {numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"Mean {numeric_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
- We loaded the dataset metadata and listed available record sets and their fields by `@id`.
- We extracted records from a main record set into a DataFrame and performed basic EDA on a numeric field (filtered, normalized, grouped).
- We visualized the numeric distribution and group aggregate.

You can adapt this template to perform further, domain-specific analyses using the `mlcroissant` record set and field `@id` references provided.